# 14. 对齐训练与偏好优化体系

预训练模型学会了“像互联网文本那样继续写”，但这不等于它已经：

- 听懂用户意图
- 遵守格式要求
- 避免有害回答
- 稳定输出人类偏好的回答

因此现代大模型通常会经历一个后训练链路：

1. SFT
2. Reward Modeling
3. RLHF / PPO 或者 DPO / ORPO / KTO 等偏好优化

## 对齐训练与偏好优化的形式化定义

对齐训练是指在预训练语言模型基础上，通过监督微调、偏好建模和策略优化等后训练步骤，使模型行为更接近人类期望的过程。其核心问题不是“让模型继续拟合文本分布”，而是“让模型生成更有用、更安全、更符合偏好的响应”。

## 输入、输出与参数化方式

            对齐训练的数据形式与预训练不同。预训练主要是无标注连续文本；对齐训练则引入：

- 指令-回答对
- 优选 / 劣选回答对
- 奖励信号
- 安全或风格标签

因此，对齐训练首先是数据范式改变，其次才是优化目标改变。

## 结构分解与信息流

            对齐训练的“架构”更准确地说是训练管线架构：

1. SFT：建立指令跟随能力
2. Reward Modeling：学习偏好排序函数
3. RLHF 或直接偏好优化：把模型推向期望行为区域

这里的重点不在于更换主干网络，而在于逐阶段改变目标函数与监督信号形式。

## 优化目标与训练机制

            SFT 使用似然最大化；Reward Model 使用排序损失；RLHF 在奖励与 KL 约束之间折中；DPO / ORPO / KTO 则尝试以更直接、更稳定的方式优化偏好。

因此，对齐训练的本质是“从 token-level likelihood 过渡到 behavior-level preference”。

## 计算复杂度、统计性质与工程代价

            RLHF 的优势在于目标灵活，但链路长、实现复杂、调参成本高。DPO 等离线方法实现更简洁，数据利用方式更稳定，但能表达的在线探索能力较弱。

在现代工业实践中，是否采用 RLHF 还是 DPO，并非理论优劣之争，而是资源、数据、工具链和目标函数形式共同决定的工程选择。

## 与相邻模型的差异

            与预训练相比，对齐训练关注模型行为而非知识覆盖。
与单纯 SFT 相比，偏好优化能进一步塑造回答风格和偏序关系。
与 RLHF 相比，DPO 等方法更易实现；与 DPO 相比，RLHF 在复杂奖励场景下更灵活。

## 先建立直觉

            预训练模型学到的是“像互联网那样继续写”，
但用户真正想要的是“按要求、按偏好、按安全边界来回答”。

对齐训练可以理解成把一个会写文本的模型，逐步变成一个更像助手的模型。

这通常不是一步完成的，而是要经历：

- 先学会模仿好回答
- 再学会区分好回答和差回答
- 再进一步朝着更符合人类偏好的方向优化

## 架构是怎么工作的

            对齐训练不是改网络结构，而是改训练目标和数据形式。

一个典型链路包含：

- SFT：用示范数据告诉模型“理想回答长什么样”
- Reward Model：学习人类偏好的排序信号
- Preference Optimization：让模型更偏向 preferred answers

所以这节的“架构”更像训练管线架构，而不是神经网络拓扑。

## 训练时到底发生了什么

            对齐训练的难点在于：人类偏好不是单一数值标签，而是复杂、主观、上下文相关的。

因此工业界才会出现多条路线：

- RLHF：更灵活，但更复杂
- DPO：更直接、更稳定
- ORPO / KTO：尝试进一步简化或改进偏好建模

学这节时，重要的不是死记公式，而是理解：
“为什么需要从 next-token prediction 走到 preference optimization？”

## 什么时候该用它

            如果你未来想做：

- 指令微调
- 对话助手
- 偏好优化
- 安全 / 有用性 / 风格控制

那这节内容就是必须掌握的基础。

## 最常见的误区

            常见误区：

1. **误以为 SFT 就等于对齐完成**
   SFT 只是第一步，很多偏好和安全问题还没有真正解决。

2. **误以为 RLHF 一定优于 DPO**
   是否更优很依赖数据、算力和工程条件。

3. **误以为偏好优化只是在刷 benchmark**
   它本质上是在改变模型行为方式。

## 1. SFT：监督微调

SFT 的目标很直接：用高质量示范数据让模型学会“该怎么回答”。

目标函数通常就是标准交叉熵：

$$
\mathcal{L}_{\mathrm{SFT}} = - \sum_t \log \pi_\theta(y_t \mid x, y_{<t})
$$

SFT 的作用：

- 让模型从 base model 进入“assistant 模式”
- 学会指令跟随、格式跟随、角色跟随
- 作为后续偏好优化的起点

## 2. Reward Model 与 Bradley-Terry

在 RLHF 中，常见做法是先训练一个奖励模型 $r_\phi(x, y)$。

如果给定同一 prompt 下的优胜回答 $y_w$ 和劣质回答 $y_l$，则常用 Bradley-Terry 形式：

$$
P(y_w \succ y_l \mid x) = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))
$$

对应损失：

$$
\mathcal{L}_{\mathrm{RM}} = -\log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))
$$

## 3. RLHF / PPO 的核心思想

RLHF 典型目标：

$$
\max_\pi \; \mathbb{E}_{y \sim \pi(\cdot \mid x)}[r(x,y)] - \beta \, \mathrm{KL}(\pi \| \pi_{\mathrm{ref}})
$$

含义：

- 让策略更偏向高奖励回答
- 但不要离参考模型偏得太远

PPO 常通过 clipped objective 保持更新稳定：

$$
\mathcal{L}_{\mathrm{PPO}} =
\mathbb{E}\left[
\min\left(
\rho_t A_t,
\mathrm{clip}(\rho_t, 1-\epsilon, 1+\epsilon) A_t
\right)
\right]
$$

其中 $\rho_t$ 是新旧策略比值，$A_t$ 是 advantage。

## 4. DPO / ORPO / KTO

### DPO

DPO 直接绕过显式 reward model + RL 过程，常见目标写作：

$$
\mathcal{L}_{\mathrm{DPO}} =
- \log \sigma \left(
\beta \left[
\log \pi_\theta(y_w \mid x) - \log \pi_\theta(y_l \mid x)
- \log \pi_{\mathrm{ref}}(y_w \mid x) + \log \pi_{\mathrm{ref}}(y_l \mid x)
\right]
\right)
$$

### ORPO

ORPO 的核心思想是把偏好优化直接并入 SFT 阶段，用 odds ratio 惩罚 rejected answer。

### KTO

KTO 从 prospect theory 视角出发，把“收益”和“损失”不对称地建模，更强调人类偏好的行为学特性。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
# 用 margin 可视化几种常见偏好优化目标。
delta = np.linspace(-5, 5, 400)
beta = 1.0

dpo_loss = -np.log(1 / (1 + np.exp(-beta * delta)))
hinge_loss = np.maximum(0, 1 - delta)
logistic_pref_loss = np.log(1 + np.exp(-delta))

plt.figure(figsize=(11, 6))
plt.plot(delta, dpo_loss, label="DPO / logistic preference", linewidth=3)
plt.plot(delta, hinge_loss, label="Margin / hinge-style", linewidth=3)
plt.plot(delta, logistic_pref_loss, label="BT logistic", linewidth=3, linestyle="--")
plt.title("偏好 margin 与损失函数的关系")
plt.xlabel("preferred minus rejected margin")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
# 构造一个玩具 preference 数据集，比较 SFT-only 和 preference-style 目标的效果。
torch.manual_seed(42)
np.random.seed(42)

prompts = ["math", "coding", "writing", "safety"] * 60
responses = ["strong", "weak"]
prompt_to_id = {name: i for i, name in enumerate(sorted(set(prompts)))}
response_to_id = {"strong": 0, "weak": 1}

X = torch.tensor([prompt_to_id[p] for p in prompts], dtype=torch.long)
y_pref = torch.tensor([0] * len(prompts), dtype=torch.long)  # 0 表示 strong 更优


class TinyPreferenceModel(nn.Module):
    def __init__(self, num_prompts, hidden_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(num_prompts, hidden_dim)
        self.scorer = nn.Linear(hidden_dim, 2)

    def forward(self, prompt_ids):
        h = self.embedding(prompt_ids)
        return self.scorer(h)


def train_sft_style(epochs=80):
    model = TinyPreferenceModel(len(prompt_to_id))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    criterion = nn.CrossEntropyLoss()
    history = []
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y_pref)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return model, history


def train_dpo_style(beta=1.0, epochs=80):
    model = TinyPreferenceModel(len(prompt_to_id))
    ref_model = TinyPreferenceModel(len(prompt_to_id))
    ref_model.load_state_dict(model.state_dict())
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    history = []

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X)
        with torch.no_grad():
            ref_logits = ref_model(X)

        logp = torch.log_softmax(logits, dim=-1)
        ref_logp = torch.log_softmax(ref_logits, dim=-1)

        chosen = logp[:, 0]
        rejected = logp[:, 1]
        ref_chosen = ref_logp[:, 0]
        ref_rejected = ref_logp[:, 1]

        margin = beta * ((chosen - rejected) - (ref_chosen - ref_rejected))
        loss = -torch.log(torch.sigmoid(margin)).mean()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return model, history

In [ ]:
sft_model, sft_history = train_sft_style()
dpo_model, dpo_history = train_dpo_style()

plt.figure(figsize=(11, 6))
plt.plot(sft_history, label="SFT-style loss")
plt.plot(dpo_history, label="DPO-style loss")
plt.title("SFT 与 DPO 风格目标的训练曲线")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
with torch.no_grad():
    sft_scores = torch.softmax(sft_model(X[:4]), dim=-1)[:, 0].numpy()
    dpo_scores = torch.softmax(dpo_model(X[:4]), dim=-1)[:, 0].numpy()

compare_df = pd.DataFrame(
    {
        "prompt": list(prompt_to_id.keys()),
        "SFT 对 preferred 的概率": sft_scores,
        "DPO 对 preferred 的概率": dpo_scores,
    }
)
compare_df

In [ ]:
compare_long = compare_df.melt(id_vars="prompt", var_name="方法", value_name="概率")
plt.figure(figsize=(10, 6))
sns.barplot(data=compare_long, x="prompt", y="概率", hue="方法")
plt.title("不同训练目标对 preferred response 的偏好程度")
plt.ylim(0, 1.0)
plt.show()

## 5. 什么时候用什么方法

- `SFT`：任何对齐链路的基础步骤，几乎总是先做
- `RLHF / PPO`：当你需要显式 reward、在线探索、复杂交互目标时更灵活
- `DPO`：当前最常见的离线偏好优化强基线，简单稳定
- `ORPO`：希望减少 reference model 依赖，训练流程更简洁时可考虑
- `KTO`：当偏好信号形式更接近“好 / 不好”二元标签时有吸引力

实践里并不存在“一个方法永远最好”。
更准确的理解是：

- 资源少、数据是离线 pairwise preference：优先 DPO / ORPO
- 有稳定 reward pipeline 和在线采样能力：RLHF / PPO 更灵活

## 6. 主要论文与官方资料

- InstructGPT / RLHF（NeurIPS 2022）:
  https://papers.nips.cc/paper_files/paper/2022/hash/b1efde53be364a73914f58805a001731-Abstract-Conference.html
- DPO（2023-05-29）:
  https://arxiv.org/abs/2305.18290
- ORPO（2024-03-12）:
  https://arxiv.org/abs/2403.07691
- KTO（2024-02-02）:
  https://arxiv.org/abs/2402.01306
- SimPO（2024-05-23）:
  https://arxiv.org/abs/2405.14734